# dl_05_land_to_bronze

Loads JSONL from `Files/_landing/<batch>/` into the bronze tables.

**This notebook holds no credentials and needs none.** That is the whole
point of the split: extraction needs an API secret and runs wherever the
secret already lives; this half needs Spark and reads files.

Use it when the source credentials are not in Key Vault yet. Once they
are, `dl_01_extract_procore` writes bronze directly and this becomes a
backfill and replay tool rather than the main path.

The load is a MERGE on `_merge_key`, so re-running is a no-op and a
partially-uploaded batch can simply be uploaded again.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

LIB = "/lakehouse/default/Files"

import glob
import json
import os

from pyspark.sql import functions as F

import fabric_common as fc

LANDING = f"{LIB}/_landing"

# Which batch to load. Empty means "the newest folder present", which is what
# you want after uploading one batch; name it explicitly to replay an older one.
BATCH = ""

In [ ]:
batches = sorted(
    d for d in glob.glob(f"{LANDING}/*") if os.path.isdir(d)
)
if not batches:
    raise RuntimeError(
        f"no batches under {LANDING}. Upload a folder produced by "
        "scripts/extract_local.py before running this."
    )

target = f"{LANDING}/{BATCH}" if BATCH else batches[-1]
files = sorted(glob.glob(f"{target}/*.jsonl"))
print(f"batch {os.path.basename(target)}: {len(files)} file(s)")
if not files:
    raise RuntimeError(f"{target} contains no .jsonl files")

In [ ]:
batch_id = fc.new_batch_id()
summary = []

for path in files:
    table = os.path.splitext(os.path.basename(path))[0]

    # Read as text and parse per line rather than spark.read.json on the folder:
    # the landing files are small, and this keeps the bronze column order fixed
    # regardless of which keys happen to appear in the first file Spark samples.
    with open(path, encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]

    if not rows:
        print(f"  {table:48s} empty")
        summary.append((table, 0))
        continue

    schema = (
        "_key string, _project_id string, _merge_key string, _source_endpoint string, "
        "_ingested_at timestamp, payload string, _batch_id string, _row_hash string"
    )
    tidy = [
        (
            r.get("_key"),
            r.get("_project_id"),
            r.get("_merge_key"),
            r.get("_source_endpoint"),
            None,  # _ingested_at is set below from the string, via a cast
            r.get("payload"),
            r.get("_batch_id"),
            r.get("_row_hash"),
        )
        for r in rows
    ]
    df = spark.createDataFrame(tidy, schema)
    # The landed timestamp is an ISO string; cast rather than trusting inference.
    df = df.withColumn("_ingested_at", F.to_timestamp(F.lit(rows[0].get("_ingested_at"))))

    written = fc.merge_delta(spark, df, table, ["_merge_key"])
    fc.log_run(spark, batch_id, "land_to_bronze", table, written)
    print(f"  {table:48s} {written:6,d} row(s)")
    summary.append((table, written))

In [ ]:
# The Controller's manual crosswalk overrides.
#
# OVERWRITE, not merge. This CSV is the authoritative list of human decisions:
# a row deleted from it means "that mapping was wrong", and merging would keep
# the retracted mapping alive forever.
# TWO PATH NAMESPACES, and mixing them is a 400 from OneLake.
#
#   /lakehouse/default/Files/...  a POSIX mount. Works for open(), os.path.
#   Files/...                     what spark.read resolves, via the ABFS driver
#                                 against the attached default lakehouse.
#
# Handing spark.read the mount path makes it request
# onelake.dfs.fabric.microsoft.com/<ws>/lakehouse/default/Files/... which is not
# a real OneLake path. Existence is checked on the mount; the read uses the
# relative form.
REFERENCE_LOCAL = f"{LIB}/reference/project_crosswalk.csv"
REFERENCE = "Files/reference/project_crosswalk.csv"

if os.path.exists(REFERENCE_LOCAL):
    crosswalk = (
        spark.read.option("header", True)
        .schema(
            "procore_project_id string, qbo_customer_id string, "
            "hubspot_deal_id string, reviewed_by string, active boolean"
        )
        .csv(REFERENCE)
    )
    crosswalk.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable("dl_bronze_reference_project_crosswalk")
    print(f"crosswalk overrides: {crosswalk.count()} row(s)")
else:
    print(f"no {REFERENCE_LOCAL} - crosswalk will rely on automatic matching only")

In [ ]:
import json
import os


def write_diag(name: str, payload: dict) -> None:
    """Structured diagnostics to Files/_diag/.

    Fabric's job API gives no per-cell detail - a failed notebook reports
    "Failed" and nothing else. Writing what happened to a file the deploy
    scripts can read back is the difference between debugging this and guessing.
    """
    os.makedirs("/lakehouse/default/Files/_diag", exist_ok=True)
    path = f"/lakehouse/default/Files/_diag/{name}.json"
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print(f"diagnostics -> {path}")

total = sum(n for _, n in summary)
print(f"\n{total:,} row(s) merged into bronze across {len(summary)} table(s)")
write_diag("land_to_bronze", {
    "batch_id": batch_id,
    "source_batch": os.path.basename(target),
    "tables": [{"table": t, "rows": n} for t, n in summary],
})